# Supervised Fine-Tuning using Pretrained SSL Encoder

---

## Notebook 3: Deepfake Classification

This notebook covers:

- Loading pretrained SSL encoder
- Building classification head
- Training loop
- Validation metrics
- Saving final classifier

Output:
models/classifier.pth

## 1. Objective

After learning invariant representations via self-supervised learning,
we now fine-tune the encoder for binary classification:

- Real (0)
- Fake (1)

Loss Function:
Binary Cross Entropy with Logits

Evaluation Metrics:
- Accuracy
- F1-score
- ROC-AUC

## Import Libraries

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm
import cv2
import glob
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

E:\conda_envs\ml_forever\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Device Configuration

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## 2. Preprocessing Pipeline

Images are:
- Resized to 224x224
- Normalized using ImageNet statistics
- Converted to tensors

In [3]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

## Dataset Class

In [4]:
class DeepfakeDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(img)

        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return img, label

## Load Training & Validation Data

In [5]:
import os
import glob

DATA_DIR = r"D:\Downloads Alt\archive\real_vs_fake\real-vs-fake"

train_real = glob.glob(os.path.join(DATA_DIR, "train/real/*.jpg"))
train_fake = glob.glob(os.path.join(DATA_DIR, "train/fake/*.jpg"))

valid_real = glob.glob(os.path.join(DATA_DIR, "valid/real/*.jpg"))
valid_fake = glob.glob(os.path.join(DATA_DIR, "valid/fake/*.jpg"))

print("Train Real:", len(train_real))
print("Train Fake:", len(train_fake))
print("Valid Real:", len(valid_real))
print("Valid Fake:", len(valid_fake))

Train Real: 50000
Train Fake: 50000
Valid Real: 10000
Valid Fake: 10000


### Create DataLoaders

In [6]:
train_real = train_real[:500]
train_fake = train_fake[:500]

valid_real = valid_real[:200]
valid_fake = valid_fake[:200]

In [7]:
train_paths = train_real + train_fake
train_labels = [0]*len(train_real) + [1]*len(train_fake)

valid_paths = valid_real + valid_fake
valid_labels = [0]*len(valid_real) + [1]*len(valid_fake)

In [8]:
train_dataset = DeepfakeDataset(train_paths, train_labels, transform)
valid_dataset = DeepfakeDataset(valid_paths, valid_labels, transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=8, shuffle=False)

In [9]:
print(len(train_paths))
print(len(valid_paths))

1000
400


## 3. Load Pretrained SSL Encoder

In [10]:
encoder = timm.create_model(
    "vit_tiny_patch16_224",
    pretrained=False,
    num_classes=0
)

encoder.load_state_dict(torch.load("models/ssl_encoder.pth"))

<All keys matched successfully>

## 4. Classification Head

Architecture:
Encoder → Linear(768 → 1)

In [11]:
class DeepfakeClassifier(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.fc = nn.Linear(192,1)

    def forward(self,x):
        features = self.encoder(x)
        return self.fc(features)

### Initialize Model

In [12]:
model = DeepfakeClassifier(encoder).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

### Training Loop

In [13]:
epochs = 2

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in tqdm(train_loader):
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}] Loss: {total_loss/len(train_loader):.4f}")

100%|████████████████████████████████████████████████████████████████████████████████| 125/125 [02:31<00:00,  1.21s/it]


Epoch [1/2] Loss: 0.6946


100%|████████████████████████████████████████████████████████████████████████████████| 125/125 [02:13<00:00,  1.07s/it]

Epoch [2/2] Loss: 0.4908


### Validation Evaluation

In [14]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

model.eval()

y_true = []
y_pred = []
y_prob = []

with torch.no_grad():
    for images, labels in valid_loader:
        images = images.to(device)

        outputs = model(images)
        probs = torch.sigmoid(outputs)

        preds = (probs > 0.5).int()

        y_pred.extend(preds.cpu().numpy().flatten())
        y_true.extend(labels.numpy().flatten())
        y_prob.extend(probs.cpu().numpy().flatten())

accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

print(f"Accuracy : {accuracy:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

Accuracy : 0.7375
F1-score : 0.7107
ROC-AUC  : 0.8282


### Save Final Classifier

In [15]:
torch.save(model.state_dict(), "models/classifier.pth")
print("Classifier saved successfully.")

Classifier saved successfully.


## Fine-Tuning Completed

We have:

- Loaded pretrained SSL encoder
- Trained supervised classifier
- Evaluated on validation data
- Saved final model

Next Step:
Cross-domain evaluation, robustness testing, and interpretability.